# Phase 5 -- Costs + Exits + Volatility Filter

Pre-registered walk-forward test of whether realistic trading costs +
smarter exit management (5 exit modes) + an ATR%-based volatility filter
make the Phase-2/4 strategy net-profitable out-of-sample, on top of entry
parameters frozen at their Phase-4-null defaults.

**Honest headline (read this first): `robust_improvement = FALSE` -- a
null result, and also the closest this program has come.** The tuned
configuration's stitched net OOS profit factor crosses breakeven for the
first time in this program (PF **1.0707**, n=228, **+$9,795** net,
expectancy **+$42.96/trade**) and passes 4 of the 5 pre-registered
conditions. It fails only condition (e): the bootstrap CI lower bound on
that PF is **0.807**, below the 1.0 confidence floor the design demands --
because the entire net edge is concentrated in one 25-trade fold. Full
detail, all mandatory disclosures, and the 5-phase program retrospective:
see `WRITEUP_PHASE5.md` in the repo root.

This notebook **loads the committed, single-shot `phase5_results.json`**
rather than re-running the walk-forward sweep (~52 seconds over a
20-combo grid x 4 folds, but still a pre-registered single-shot run). The
pre-registration freeze means that JSON *is* the result -- re-running it
here would defeat the point of a single-shot, audited run (`config_hash` +
`git_sha` are recorded in the file precisely so a re-run can be told apart
from the original).

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # non-interactive; this notebook only displays pre-rendered PNGs
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd

# Make relative paths (phase5_results.json, charts/...) resolve correctly
# whether Jupyter was launched from the repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
if not (Path.cwd() / "phase5_results.json").exists():
    raise RuntimeError(f"expected repo root, got {Path.cwd()}")
print(f"Working directory: {Path.cwd()}")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 2. Load the committed results

`phase5_results.json` is the single-shot, pre-registered output of
`python run_phase5.py`. It carries its own audit trail: a SHA-256 hash of
the *whole* frozen design -- grid, folds, `MIN_IS_TRADES`,
`MIN_OOS_TRADES`, objective, the vol-filter definition, ATR method, exit
sequencing, cost constants, `TIME_STOP_ET` (`config_hash`) -- and the git
commit the frozen design was built on (`git_sha`). Note: `git_sha` here
points at the **design-freeze parent commit**, not the commit that added
the results file itself -- this project's single-shot workflow runs
against the frozen design and commits the results afterward. The
`config_hash` reproducing byte-identically is the actual proof this ran
against the pre-registered design unmodified.

In [ ]:
with open("phase5_results.json") as f:
    results = json.load(f)

pd.Series({
    "config_hash": results["config_hash"],
    "git_sha": results["git_sha"],
    "grid_size": results["grid_size"],
    "min_is_trades": results["min_is_trades"],
    "min_oos_trades": results["min_oos_trades"],
    "objective": results["objective"],
    "run_seconds": results["run_seconds"],
    "n_folds": len(results["folds"]),
})

In [ ]:
pd.Series(results["cost_model"])

## 3. The pre-registered verdict -- all 5 conditions

The success rule (`docs/superpowers/plans/2026-07-13-phase5-exits-costs-volfilter.md`)
was fixed **before** the run: a positive ("Phase 5 robustly rescues net
profitability") verdict requires **all five** conditions below.
`robust_improvement` is the compound AND of all five -- and it is
`False`, because condition (e) alone fails.

In [ ]:
sr = results["success_rule"]

rule_rows = [
    {
        "condition": "(a) tuned net stitched-OOS PF > 1.0",
        "value": round(sr["condition_a_tuned_net_pf_gt_1"]["tuned_stitched_net_oos_pf"], 4),
        "threshold": "> 1.0",
        "passes": sr["condition_a_tuned_net_pf_gt_1"]["passes"],
    },
    {
        "condition": "(b) tuned - base net PF margin",
        "value": round(sr["condition_b_margin_gte_0_10"]["margin"], 4),
        "threshold": ">= 0.10",
        "passes": sr["condition_b_margin_gte_0_10"]["passes"],
    },
    {
        "condition": "(c) tuned beats base net PF in >= 3/4 folds",
        "value": f"{sr['condition_c_wins_3_of_4_folds']['folds_tuned_beats_base']}/{sr['condition_c_wins_3_of_4_folds']['total_folds']}",
        "threshold": ">= 3/4",
        "passes": sr["condition_c_wins_3_of_4_folds"]["passes"],
    },
    {
        "condition": "(d) tuned net PF >= p75 of 20-combo null",
        "value": f"{round(sr['condition_d_beats_p75_combo_null']['tuned_stitched_net_oos_pf_capped'], 4)} >= {round(sr['condition_d_beats_p75_combo_null']['p75_of_20_combos_stitched_net_oos_pf'], 4)}",
        "threshold": ">= p75",
        "passes": sr["condition_d_beats_p75_combo_null"]["passes"],
    },
    {
        "condition": "(e) n>=60 AND bootstrap CI lower (5th pct) > 1.0",
        "value": f"n={sr['condition_e_oos_sample_gate']['n_oos_trades']}, CI_lo={round(sr['condition_e_oos_sample_gate']['bootstrap_ci_lower_5th_pct'], 4)}",
        "threshold": "n>=60 & CI_lo>1.0",
        "passes": sr["condition_e_oos_sample_gate"]["passes"],
    },
]
rule_df = pd.DataFrame(rule_rows)
rule_df

In [ ]:
print("robust_improvement =", sr["robust_improvement"])

Four of five conditions pass -- the point estimate clears profitability
(a), clears the margin-over-base bar (b), wins 3 of 4 folds (c), and beats
the 75th percentile of the 20-combo selection-luck null (d). Condition (e)
-- the only failure -- exists specifically to catch a result this
concentrated: n=228 trades comfortably clears the 60-trade floor, but a
1,000-resample bootstrap (fixed seed 42) over the stitched net-OOS PF puts
the 5th-percentile lower bound at 0.807, well under 1.0 (95th-percentile
upper bound: 1.394). The point estimate says "profitable"; the resampling
distribution says "cannot rule out a below-1.0 strategy that got lucky in
this sample." Section 5 below shows exactly why the CI is this wide.

## 4. Stitched OOS -- tuned vs. base (2024-25, never seen during selection)

In [ ]:
stitched = results["stitched_oos"]["all_folds"]
stitched_df = pd.DataFrame(stitched).T
stitched_df.index.name = "side"
stitched_df

Tuned crosses breakeven **net of costs** for the first time in this
program: PF 1.0707, +$9,795 net P&L, +$42.96 expectancy/trade, on 228
trades. Base (`fixed_1_5R`, no vol filter, same costs) stays
net-unprofitable over 401 trades: PF 0.9006, -$25,587.50, -$63.81/trade.
Profit factor and expectancy are **co-reported deliberately**: PF alone is
a confounded metric across heterogeneous exit shapes (`trail_swing` has
unbounded upside; `fixed_1_5R` caps at 1.5R), so expectancy-per-trade is
the more legible, less-confounded number to anchor on.

## 5. Per-fold detail

In [ ]:
fold_rows = []
for f in results["folds"]:
    fold_rows.append({
        "fold": f["fold_index"],
        "test_window": f"{f['test_start'][:10]} .. {f['test_end'][:10]}",
        "selected exit / vol": f"{f['selected_exit_mode']} / {f['selected_vol_filter']}",
        "fallback_used": f["fallback_used"],
        "is_net_pf": round(f["is_net_pf"], 4),
        "is_n": f["is_n"],
        "oos_tuned_pf": round(f["oos_net_metrics"]["net_profit_factor"], 4),
        "oos_tuned_n": f["oos_net_metrics"]["n_trades"],
        "oos_tuned_pnl": f["oos_net_metrics"]["net_total_pnl"],
        "oos_base_pf": round(f["oos_net_base_metrics"]["net_profit_factor"], 4),
        "oos_base_n": f["oos_net_base_metrics"]["n_trades"],
        "tuned_wins": f["oos_net_metrics"]["net_profit_factor"] > f["oos_net_base_metrics"]["net_profit_factor"],
        "selected_oos_percentile": f["selected_oos_percentile"],
    })
fold_df = pd.DataFrame(fold_rows)
fold_df

Fold 1 is the one loss (tuned PF 0.723 on 30 trades vs. base's 0.880 on
102) -- its selected combo (`partial_1R`/`p50`) is the same one selected in
Fold 2, but Fold 1's in-sample PF that picked it (0.919) was itself under
1.0: "least-bad of a weak in-sample field," not a real edge. Folds 2-4 all
beat base. Fold 4's margin is enormous (PF 2.248 vs. 1.069) -- which is
exactly the fold examined in the next section.

## 6. The Fold-4 concentration problem -- why condition (e) is right to say no

This is the central honesty finding of Phase 5.

In [ ]:
f4 = results["folds"][3]
other_folds_pnl = sum(f["oos_net_metrics"]["net_total_pnl"] for f in results["folds"] if f["fold_index"] != 3)
stitched_total = results["stitched_oos"]["all_folds"]["tuned"]["net_total_pnl"]

pd.Series({
    "fold_4_net_pnl": f4["oos_net_metrics"]["net_total_pnl"],
    "fold_4_n_trades": f4["oos_net_metrics"]["n_trades"],
    "fold_4_expectancy_per_trade": round(f4["oos_net_metrics"]["net_expectancy"], 2),
    "other_3_folds_net_pnl": other_folds_pnl,
    "stitched_total_net_pnl": stitched_total,
})

**Fold 4 alone contributes +\$18,845 in net P&L on just 25 trades** --
more than the entire stitched result (+\$9,795). **The other three folds
combined net -\$9,050.** Strip out Fold 4 and the "net-profitable
strategy" headline disappears. `trail_swing`'s unbounded upside is doing
exactly what it's designed to do when a trend cooperates, and Fold 4
(2025-07 to 2025-12) evidently had trending conditions this combination
caught well -- but a result concentrated almost entirely in one fold out
of four is exactly what a naive point estimate would over-claim on. This
is precisely what the bootstrap CI (condition e) is built to catch:
resampling the 228 stitched trades routinely draws samples that
under-represent Fold 4's concentrated run, pulling the 5th-percentile
bootstrap PF below 1.0. The pre-registered rule doing its job here is the
point, not a technicality to explain away.

## 7. Cost sensitivity -- does the verdict survive 0x -> 2x?

In [ ]:
cs = results["cost_sensitivity"]
cs_rows = []
for key in ["0x", "1x", "2x"]:
    row = cs[key]
    cs_rows.append({
        "multiplier": row["multiplier"],
        "tuned_pf": round(row["tuned"]["net_profit_factor"], 4),
        "tuned_total_pnl": row["tuned"]["net_total_pnl"],
        "base_pf": round(row["base"]["net_profit_factor"], 4),
        "base_total_pnl": row["base"]["net_total_pnl"],
        "tuned_pf_gt_1": row["tuned_pf_gt_1"],
        "tuned_beats_base_margin_ge_0_10": row["tuned_beats_base_margin_ge_0_10"],
    })
cs_df = pd.DataFrame(cs_rows).set_index("multiplier")
cs_df

The **qualitative** verdict is stable across the tested range: tuned
stays PF > 1 and base stays PF < 1 at 0x, 1x, and 2x costs. But the margin
compresses steadily as costs rise (tuned net P&L: +\$13,405 -> +\$9,795 ->
+\$6,185) -- the expected direction, and a reminder that this is a thin
margin over breakeven even before the statistical-confidence question is
asked. Disclosure: 1-tick stop slippage (the 1x case) is optimistic for
NQ in a fast market -- the 2x band is the more realistic pessimistic
case.

## 8. Selection-luck null -- the 20 combos' own stitched net-OOS PF

Every one of the 20 grid combos (5 exit modes x 4 vol filters) has its OOS
trades tracked across all 4 folds independently of which combo each fold
actually selected, giving each of the 20 a full stitched-OOS PF. This is
the distribution condition (d) compares the adaptive pick against.

In [ ]:
combo_df = pd.DataFrame(results["combo_null_distribution"])
combo_df = combo_df.sort_values("stitched_net_oos_pf", ascending=False).reset_index(drop=True)
tuned_pf = sr["condition_a_tuned_net_pf_gt_1"]["tuned_stitched_net_oos_pf"]
combo_df["beats_adaptive_pick"] = combo_df["stitched_net_oos_pf"] > tuned_pf
combo_df

In [ ]:
n_beat = int(combo_df["beats_adaptive_pick"].sum())
print(f"Adaptive (per-fold-selected) pick's stitched net-OOS PF: {tuned_pf:.4f}")
print(f"Combos that would have stitched HIGHER on their own:      {n_beat} of {len(combo_df)}")
print(f"p75 of the 20 combos: {sr['condition_d_beats_p75_combo_null']['p75_of_20_combos_stitched_net_oos_pf']:.4f}")
print(f"median of the 20 combos: {sr['condition_d_beats_p75_combo_null']['median_of_20_combos_stitched_net_oos_pf']:.4f}")

The adaptive pick clears the p75 null -- condition (d) passes -- but only
just: **5 of the 20 fixed combos** (`time_stop`/p75, `trail_swing`/p50,
`time_stop`/p50, `trail_swing`/p75, `fixed_1_5R`/p75) would have stitched
*higher* on their own than the per-fold adaptive selection did. A handful
of the simplest fixed combinations -- including the plain `fixed_1_5R`
baseline exit paired with the `p75` filter -- do about as well or better
than re-selecting exit/filter every 6 months. The same caution Phase 4
raised about its own 144-combo grid applies here at smaller scale.

## 9. Eligibility -- how thin did the filtered arms get?

The `MIN_IS_TRADES=50` floor is evaluated on each fold's **pre-filter**
in-session signal count (identical across all 20 combos), so tighter vol
filters are never auto-disqualified by the very filter under test. But
realized trade counts still shrink hard as the filter tightens.

In [ ]:
elig_rows = []
for f in results["folds"]:
    for c in f["combo_eligibility"]:
        elig_rows.append({
            "fold": f["fold_index"],
            "exit_mode": c["exit_mode"],
            "vol_filter": c["vol_filter"],
            "is_realized": c["is_realized_trades"],
            "oos_realized": c["oos_realized_trades"],
        })
elig_df = pd.DataFrame(elig_rows)
elig_summary = elig_df.groupby("vol_filter")[["is_realized", "oos_realized"]].mean().round(1)
elig_summary = elig_summary.reindex(["off", "p25", "p50", "p75"])
elig_summary.columns = ["avg_is_realized_trades", "avg_oos_realized_trades"]
elig_summary

`p75` cuts realized OOS trades to roughly a third of `off` (averaged
across all 5 exit modes x 4 folds -- 20 combos per row above). The
selected combos in this study land on `p50` (3 of 4 folds) or `off` (1 of
4), never the thinnest `p75` tier -- at least consistent with selection
not chasing an over-filtered, too-thin sample, though that is a
description of what happened, not a guarantee it would happen again.

## 10. Charts

Generated by `run_phase5.py` and committed under `charts/`; loaded here
from disk to narrate alongside the tables above (the `Agg` backend set in
Section 1 makes this notebook's own plotting non-interactive and safe to
run headless -- these cells only display pre-rendered PNGs, no live
plotting of results happens here).

### Stitched OOS equity curve -- tuned (net) vs. base (net) vs. tuned (gross)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(mpimg.imread("charts/phase5_equity_curve.png"))
ax.axis("off")
plt.show()

### Per-fold net OOS profit factor -- tuned vs. base

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(mpimg.imread("charts/phase5_oos_pf_per_fold.png"))
ax.axis("off")
plt.show()

### Selected exit mode + volatility filter per fold (stability)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.imshow(mpimg.imread("charts/phase5_selected_exit_vol_stability.png"))
ax.axis("off")
plt.show()

### Cost sensitivity -- stitched net total P&L at 0x / 1x / 2x costs

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(mpimg.imread("charts/phase5_cost_sensitivity.png"))
ax.axis("off")
plt.show()

### Selection-luck null -- each of the 20 combos' own stitched net-OOS PF

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.imshow(mpimg.imread("charts/phase5_combo_null.png"))
ax.axis("off")
plt.show()

## 11. Honest conclusion

- **`robust_improvement = False`.** Four of five pre-registered conditions
  pass; only condition (e) -- the bootstrap-CI confidence gate -- fails.
- **This is the closest the program has come.** For the first time, the
  stitched net OOS point estimate crosses breakeven (PF 1.0707, +\$9,795,
  +\$42.96/trade expectancy) using smarter exits (`trail_swing`,
  `partial_1R`) and a `p50` ATR% volatility filter, on top of the
  Phase-4-null default entry.
- **But it is not a certified edge.** The bootstrap CI lower bound (0.807)
  sits below 1.0, and the entire net edge traces to one 25-trade fold
  (+\$18,845 in Fold 4 vs. -\$9,050 net across the other three folds
  combined). The pre-registered confidence gate exists specifically to
  catch a result this concentrated, and it does its job.
- **Mandatory disclosures** (full detail in `WRITEUP_PHASE5.md`): this is
  the third pre-registered pass over the same 3-year series (after
  Phase 2's build/validate and Phase 4's 144-combo sweep); it is
  conditional on the Phase-4-null default entry, not a fresh entry
  re-tune; the 3R partial-remainder target, the \$5/1-tick cost model, and
  `TIME_STOP_ET=11:00` are arbitrary pre-registered choices (1-tick stop
  slippage is optimistic -- the 2x cost band is the realistic case); n=4
  folds is descriptive with overlapping train windows; `trail_swing` can
  legitimately exit slightly negative even after +1R activates (spec-
  correct, confirmed in the OOS trade data); and the recorded `git_sha` is
  the design-freeze parent commit, with `config_hash` reproducing
  byte-identically as the actual provenance proof.
- **The honest framing:** a promising signal, clearly and quantitatively
  distinguished from a proven one, by machinery built before the result
  was known. See `WRITEUP_PHASE5.md`'s closing "Program epilogue" for the
  full 5-phase retrospective.